# Forecasting Pipeline Test

Goal:
1. Parse the dataset into one train/test split.
2. Train one fast model first: Linear Regression.
3. Predict on held-out test data.
4. Compute RMSE, naive RMSE, RMSSE, eta, and horizon-wise RMSE.

In [5]:
# Cell 1: Basic imports

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make sure the notebook can import from the current repo folder
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

print("Current folder:", repo_root)
print("Python executable:", sys.executable)


Current folder: /Users/oscarzhang/Desktop/Project Folder/fmri_forecasting
Python executable: /Users/oscarzhang/Desktop/Project Folder/fmri_forecasting/venv/bin/python


In [6]:
# Cell 2: Import project helpers

from utils.parse_data import parse_dataset

from utils.training import (
    train_forecasting_model,
    predict_forecasting_model,
    compute_rmse,
    compute_naive_rmse,
    compute_rmsse,
    compute_eta,
    horizon_rmse,
)

from sklearn.linear_model import LinearRegression

print("Imports loaded successfully.")


Imports loaded successfully.


In [ ]:
# Cell 3: Set dataset path and forecasting parameters

ROOT_DIR = Path("data") / "pooled_stratified_share"

# Forecasting setup
M = 50       # number of past timepoints used as input
H = 3        # number of future timepoints predicted
STRIDE = 1   # sliding window stride

print("Dataset folder:", ROOT_DIR)
print("Exists?", ROOT_DIR.exists())


Dataset folder: data/pooled_stratified_share
Exists? True


In [8]:
# Cell 4: Parse dataset into train/test arrays

X_train, Y_train, X_test, Y_test, device = parse_dataset(
    root_dir=ROOT_DIR,
    M=M,
    H=H,
    normalize=True,
    stride=STRIDE,
    test_ratio=0.2,
    test_subjects=None,
    random_state=42,
    verbose=True,
)

print("\nReturned arrays:")
print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_test :", X_test.shape)
print("Y_test :", Y_test.shape)
print("Device :", device)


Using dataset path: data/pooled_stratified_share
Loading dataset with fixed schema settings...
Scanning and loading dataset (optimized fixed schema mode)...
Loaded: 2144 runs from 6 subjects
Subjects: ['subjOenz4SHyaO', 'subjQX6R7rImeb', 'subjoVC0sHUB_P', 'subjqFdsBbuiHF', 'subjtySNgRhV65', 'subjxpYwO4azeZ']
Successfully loaded runs: 2144
Loading time: 1.07 seconds
Device: cpu
Normalizing dataset...
Building sliding windows...
Splitting by subject...
Train subjects: ['subjqFdsBbuiHF', 'subjoVC0sHUB_P', 'subjxpYwO4azeZ', 'subjtySNgRhV65']
Test subjects : ['subjQX6R7rImeb', 'subjOenz4SHyaO']
Train runs: 1440
Test runs : 704
Final shapes - Train: (250560, 50, 19), (250560, 3, 19) | Test: (122496, 50, 19), (122496, 3, 19)

Returned arrays:
X_train: (250560, 50, 19)
Y_train: (250560, 3, 19)
X_test : (122496, 50, 19)
Y_test : (122496, 3, 19)
Device : cpu


In [9]:
# Cell 5: Quick sanity checks

assert X_train.ndim == 3, "X_train should have shape (N_train, M, ROI)"
assert Y_train.ndim == 3, "Y_train should have shape (N_train, H, ROI)"
assert X_test.ndim == 3, "X_test should have shape (N_test, M, ROI)"
assert Y_test.ndim == 3, "Y_test should have shape (N_test, H, ROI)"

assert X_train.shape[1] == M, "X_train window length does not match M"
assert Y_train.shape[1] == H, "Y_train horizon length does not match H"
assert X_train.shape[2] == Y_train.shape[2], "Train X/Y ROI count mismatch"
assert X_test.shape[2] == Y_test.shape[2], "Test X/Y ROI count mismatch"

print("Sanity checks passed.")
print("Number of ROIs:", X_train.shape[2])


Sanity checks passed.
Number of ROIs: 19


## Linear Regression Model

For this local notebook, I'll only use linear Regression. Will use all 4 on lab server

`training.py` automatically flattens:

```python
X_train: (N, M, ROI) -> (N, M*ROI)
Y_train: (N, H, ROI) -> (N, H*ROI)
```

Then after prediction, it reshapes predictions back to `(N, H, ROI)`.


In [10]:
# Cell 6: Build and train Linear Regression model

model = LinearRegression()

model = train_forecasting_model(
    model=model,
    X_train=X_train,
    Y_train=Y_train,
    device=device,
)

print("Linear Regression training complete.")


Linear Regression training complete.


In [11]:
# Cell 7: Predict on test data

preds, targets = predict_forecasting_model(
    model=model,
    X=X_test,
    Y=Y_test,
    device=device,
)

print("Predictions complete.")
print("preds shape  :", preds.shape)
print("targets shape:", targets.shape)


Predictions complete.
preds shape  : (122496, 3, 19)
targets shape: (122496, 3, 19)


In [12]:
# Cell 8: Compute metrics

model_rmse = compute_rmse(targets, preds)
naive_rmse = compute_naive_rmse(X_test, Y_test)
model_rmsse = compute_rmsse(targets, preds, X_train)
eta = compute_eta(targets, preds)
horizon_scores = horizon_rmse(targets, preds)

print("\nMain metrics:")
print(f"Model RMSE : {model_rmse:.6f}")
print(f"Naive RMSE : {naive_rmse:.6f}")
print(f"RMSSE      : {model_rmsse:.6f}")
print(f"Eta        : {eta:.6f}")
print(f"Beat naive?: {'YES' if model_rmse < naive_rmse else 'NO'}")



Horizon-wise RMSE:
  Step 1 RMSE: 0.701964
  Step 2 RMSE: 0.825764
  Step 3 RMSE: 0.861641

Main metrics:
Model RMSE : 0.799388
Naive RMSE : 0.991941
RMSSE      : 1.008540
Eta        : 0.122473
Beat naive?: YES


In [13]:
# Cell 9: Save metrics to CSV

results = pd.DataFrame([
    {
        "model": "Linear Regression",
        "M": M,
        "H": H,
        "stride": STRIDE,
        "model_rmse": model_rmse,
        "naive_rmse": naive_rmse,
        "rmsse": model_rmsse,
        "eta": eta,
        "beat_naive": model_rmse < naive_rmse,
    }
])

results_path = "linear_regression_simple_results.csv"
results.to_csv(results_path, index=False)

print("Saved results to:", results_path)
results


Saved results to: linear_regression_simple_results.csv


,model,M,H,stride,model_rmse,naive_rmse,rmsse,eta,beat_naive
0,Linear Regression,50,3,1,0.799388,0.991941,1.00854,0.122473,True
